# Лабораторная 1. HTTP-запросы
Две задачи: получить список репозиториев GitHub и выполнить запрос к API с авторизацией.
Второй источник - OpenWeather. Ниже показаны сохранённые ответы. Команды нового запуска находятся в README.

In [1]:
import html
import json
import os
from pathlib import Path
from IPython.display import HTML, display

def show_table(rows, fields):
    head = "".join("<th>" + html.escape(field) + "</th>" for field in fields)
    body = "".join("<tr>" + "".join("<td>" + html.escape(str(row.get(field, ""))) + "</td>"
                                    for field in fields) + "</tr>" for row in rows)
    display(HTML("<table><thead><tr>" + head + "</tr></thead><tbody>" + body + "</tbody></table>"))

## 1. Запрос к GitHub
Метод GET, адрес `https://api.github.com/users/onedanoken/repos`.
Параметры `per_page=30` и `sort=updated` задают размер порции и сортировку.
В скрипте JSON сохраняется только после ответа со статусом 200.

In [2]:
print(Path("task1/get_repos_request.py").read_text(encoding="utf-8"))

import requests
import json

username = "onedanoken"
url = f"https://api.github.com/users/{username}/repos"

params = {
    "per_page": 30,
    "sort": "updated"
}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    
    filename = f"{username}_repos.json"
    with open(filename, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)
        
    print(f"Found {len(data)} repos.")
    print(f"File: {filename}")
else:
    print(f"Error: {response.status_code}: {response.text}")


## 2. Сохранённый результат GitHub
JSON содержит массив объектов. Для просмотра выбраны название, ссылка и число звёзд.
Это сохранённый ответ, его число записей может отличаться от нового запроса.

In [3]:
repos = json.loads(Path("task1/onedanoken_repos.json").read_text(encoding="utf-8"))
print("Репозиториев:", len(repos))
show_table(repos[:5], ["full_name", "html_url", "stargazers_count"])

Репозиториев: 15


full_name,html_url,stargazers_count
onedanoken/macroscop_demo,https://github.com/onedanoken/macroscop_demo,1
onedanoken/program_product,https://github.com/onedanoken/program_product,0
onedanoken/ros_test,https://github.com/onedanoken/ros_test,0
onedanoken/YLabTask,https://github.com/onedanoken/YLabTask,0
onedanoken/timeseries_homeworks,https://github.com/onedanoken/timeseries_homeworks,0


## 3. Запрос с API-ключом
OpenWeather принимает город, единицы измерения, язык и API-ключ.
Ключ берётся из `OPENWEATHER_API_KEY`. Его значение не выводится и не хранится в репозитории.

In [4]:
print(Path("task2/get_weather_request.py").read_text(encoding="utf-8"))
print("Ключ задан в окружении:", bool(os.environ.get("OPENWEATHER_API_KEY")))

import requests
import json
import os

API_KEY = os.environ["OPENWEATHER_API_KEY"]
CITY = "Пермь"

url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": CITY,
    "appid": API_KEY,
    "units": "metric",
    "lang": "ru"
}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    
    temp = data["main"]["temp"]
    description = data["weather"][0]["description"]
    print(f"Погода: {temp}°C, {description}")
    
    filename = "weather.json"
    with open(filename, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)
        
    print(f"File: {filename}")
else:
    print(f"Error {response.status_code}: {response.text}")
Ключ задан в окружении: False


## 4. Сохранённый ответ OpenWeather
Из вложенных объектов извлекаются город, температура, описание погоды и влажность.
Это пример разбора сохранённого ответа, а не текущая погода.

In [5]:
weather = json.loads(Path("task2/weather.json").read_text(encoding="utf-8"))
show_table([{
    "Город": weather.get("name"),
    "Температура, °C": weather["main"]["temp"],
    "Описание": weather["weather"][0]["description"],
    "Влажность, %": weather["main"]["humidity"],
}], ["Город", "Температура, °C", "Описание", "Влажность, %"])

Город,"Температура, °C",Описание,"Влажность, %"
Пермь,18.7,пасмурно,68


## Вывод
В обоих заданиях запрос возвращает JSON. Скрипты проверяют HTTP-статус, разбирают ответ и сохраняют файл.
Для публичных репозиториев GitHub ключ не требуется; для OpenWeather используется API-ключ.